In [19]:
import anndata as ad

st_data = ad.read_h5ad('/compute/oven-0-13/jwinchel/processed_adata/NSCLC_ST_withprobs.h5ad')
sc_data = ad.read_h5ad("/work/magroup/skrieger/Syndri/data/label_transfer/SC_NSCLC_labeltransfer_admixed_10reps.h5ad")

In [20]:

st_data.obs['cell_type'] = st_data.obs['cell_type'].replace({"T CD4 naive": "T CD4+", 
                                                                "T CD4 memory": "T CD4+", 
                                                                "T CD8 naive": "T CD8+", 
                                                                "T CD8 memory": "T CD8+",
                                                                "pDC":"DC",
                                                                "mDC":"DC"})
st_data.obs['cell_type'] = st_data.obs['cell_type'].apply(lambda x: "tumor" if x.startswith("tumor") else x)                                                                

st_data.obs['cell_type'].value_counts()

/tmp/ipykernel_1149192/3177676375.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  st_data.obs['cell_type'] = st_data.obs['cell_type'].replace({"T CD4 naive": "T CD4+",


cell_type
tumor          18600
fibroblast     13151
neutrophil      9925
T CD4+          7692
endothelial     7613
macrophage      7409
B-cell          7207
DC              7088
plasmablast     7079
epithelial      4098
mast            2462
T CD8+          2020
monocyte        1843
NK              1668
Treg             147
Name: count, dtype: int64

In [21]:
st_data.uns['cell_type_dict'] = sc_data.uns['cell_type_dict']

In [22]:
st_data.obs[['ADM_ID_0', 'ADM_ID_1', 'ADM_ID_2']] = None

st_data = st_data[st_data.obs['slide_ID_numeric']==1]

In [23]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

def compute_spatial_knn(adata, k=3):
    
    coords = adata.obs[["CenterX_global_px", "CenterY_global_px"]].values
    
    nn = NearestNeighbors(n_neighbors=k+1, algorithm="kd_tree", metric="euclidean")
    nn.fit(coords)
    
    distances, indices = nn.kneighbors(coords)
    
    # remove self neighbor (first column)
    distances = distances[:, 1:]
    indices = indices[:, 1:]
    
    return distances, indices

dists, indices = compute_spatial_knn(st_data)

st_data.obs[['ADM_ID_0', 'ADM_ID_1', 'ADM_ID_2']] = indices.astype(str)

/tmp/ipykernel_1149192/319805141.py:21: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  st_data.obs[['ADM_ID_0', 'ADM_ID_1', 'ADM_ID_2']] = indices.astype(str)


In [24]:
# relabel individual tumor cell types as general "tumor"
st_data.obs['cell_type'] = st_data.obs['cell_type'].apply(lambda x: 'tumor' if x.startswith('tumor') else x)
st_data.obs['cell_type'].value_counts()

cell_type
tumor          18600
fibroblast     13151
neutrophil      9925
T CD4+          7692
endothelial     7613
macrophage      7409
B-cell          7207
DC              7088
plasmablast     7079
epithelial      4098
mast            2462
T CD8+          2020
monocyte        1843
NK              1668
Treg             147
Name: count, dtype: int64

In [25]:
st_data.obs

,fov,cell_ID,tissue,sdimx,sdimy,Area,AspectRatio,Width,Height,Mean.CD298,...,prop_tumor_in_100_neighbors,cell_ID_short,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px,split,ADM_ID_0,ADM_ID_1,ADM_ID_2
0,1.0,1.0,Lung5_Rep1,28.78886,-3.28806,1259.0,1.34,47.0,35.0,3473.0,...,0.09,1.0,1027,3631,4215.888889,158847.666667,val,9,20054,1913
1,1.0,2.0,Lung5_Rep1,29.12672,-3.29040,3723.0,1.45,87.0,60.0,3895.0,...,0.00,2.0,2904,3618,6092.888889,158834.666667,train,22,41,20049
2,1.0,3.0,Lung5_Rep1,29.32868,-3.28878,2010.0,1.62,68.0,42.0,2892.0,...,0.01,3.0,4026,3627,7214.888889,158843.666667,val,34,1903,28
3,1.0,4.0,Lung5_Rep1,29.36540,-3.29418,3358.0,0.47,48.0,102.0,6189.0,...,0.00,4.0,4230,3597,7418.888889,158813.666667,val,1904,4,1921
4,1.0,5.0,Lung5_Rep1,29.37044,-3.28842,1213.0,1.00,38.0,38.0,8138.0,...,0.00,5.0,4258,3629,7446.888889,158845.666667,val,3,20050,1905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97997,30.0,4750.0,Lung5_Rep1,32.87952,-0.65358,1614.0,2.06,64.0,31.0,889.0,...,0.00,4750.0,1864,17,26941.777778,173483.666667,val,74930,97102,74965
97998,30.0,4751.0,Lung5_Rep1,32.89248,-0.65430,1346.0,2.83,68.0,24.0,577.0,...,0.00,4751.0,1936,13,27013.777778,173479.666667,val,97151,97134,74931
97999,30.0,4752.0,Lung5_Rep1,33.39126,-0.65376,1336.0,1.86,54.0,29.0,309.0,...,0.00,4752.0,4707,16,29784.777778,173482.666667,val,78494,74999,97114
98000,30.0,4753.0,Lung5_Rep1,33.44832,-0.65466,803.0,2.58,49.0,19.0,834.0,...,0.00,4753.0,5024,11,30101.777778,173477.666667,val,74945,97979,74955


In [26]:
st_data.write_h5ad('/compute/oven-0-13/jwinchel/processed_adata/NSCLC_ST_withneighbors.h5ad')